In [1]:
import ast
import copy
import gc
import os
import random
import shutil
import subprocess
import sys
from pathlib import Path

import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torchvision
import torch.nn as nn
from collections import defaultdict
from google.colab import drive
from PIL import Image
from sklearn.model_selection import StratifiedKFold, train_test_split
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torch.nn.modules.activation import LeakyReLU
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score
from sklearn.base import clone

c:\Users\annab\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ModuleNotFoundError: No module named 'torchvision'

In [ ]:
os.environ['KAGGLE_USERNAME'] = "annabellerondeau"
os.environ['KAGGLE_API_TOKEN'] = "KGAT_65f3340fa30461c90995a50a1376b582"
path = kagglehub.competition_download('fall-2026-ecse-551-task-1')

print("Path to competition files:", path)
print(os.listdir(path))

Data Preprocessing

In [ ]:
# Set-up, add labels and split the data 70:15:15 for training:validate:test
def seed_everything(seed=42):
  # Set seed for Pytorch, Numpy to have reproducible random operations
  random.seed(seed)
  os.environ["PYTHONHASHSEED"] = str(seed)
  np.random.seed(seed)

  torch.manual_seed(seed)
  torch.cuda.manual_seed_all(seed)

seed_everything(42)

train_df = pd.read_csv(os.path.join(path, "train.csv"))
test_df = pd.read_csv(os.path.join(path, "test.csv"))

# Define sorted unique class names to keep the same index assignments across runs
classes = sorted(train_df["label_name"].unique())
label_to_index = {}
index_to_label = {}

for index, class_name in enumerate(classes):
  label_to_index[class_name] = index
for class_name, index in label_to_index.items():
  index_to_label[index] = class_name

num_classes = len(classes)

train_df = train_df.copy()
train_df["label"] = train_df["label_name"].map(label_to_index)

# 70% Train, 15% Val, 15% Local Test (stratified)
training_df, temp_df = train_test_split(train_df,test_size=0.30,stratify=train_df["label"],random_state=42,)
val_df, local_test_df = train_test_split(temp_df,test_size=0.50,stratify=temp_df["label"],random_state=42,)


In [ ]:
# Perform transforms
class AddGaussianNoise:
  def __init__(self, mean=0.0, std=0.03, p=0.3):
    self.mean = mean
    self.std = std
    self.p = p

  def __call__(self, tensor):
    # Adds zero-mean Gaussian noise to a tensor image
    if torch.rand(1).item() < self.p:
        noise = torch.randn_like(tensor) * self.std + self.mean
        return tensor + noise
    return tensor


IMG_SIZE = 224
# from ImageNet Reference values
MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]

# For training we augment and perform pre-processing
# None, light and heavy transforms can be used during training

train_transform_none = transforms.Compose([
  transforms.Resize((IMG_SIZE, IMG_SIZE)), # Change resolution before cropping/standardizing to IMG_SIZE
  transforms.ToTensor(), #convert to tensor
  transforms.Normalize(mean=MEAN, std=STD),
])

train_transform_light = transforms.Compose([
  transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)), # cropping
  transforms.RandomHorizontalFlip(p=0.5), # flip the image
  transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15, hue=0.03),  # change the colors
  transforms.ToTensor(), #convert to tensor
  transforms.Normalize(mean=MEAN, std=STD), # normalize channels
])

train_transform_heavy = transforms.Compose([
  transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)), # cropping
  transforms.RandomHorizontalFlip(p=0.5), # flip the image
  transforms.RandomRotation(degrees=15), #rotate
  transforms.RandomAffine(degrees=0, shear=10), # warping: shear
  transforms.RandomPerspective(distortion_scale=0.15, p=0.2), # warping: scale
  transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),  # change the colors
  transforms.RandomGrayscale(p=0.1), #change to grayscale
  transforms.ToTensor(), #convert to tensor
  AddGaussianNoise(mean=0.0, std=0.03, p=0.3), #add gaussian noise
  transforms.Normalize(mean=MEAN, std=STD), # normalize channels
])

# Keep validation / local test / Kaggle test deterministic (only preprocessing)
eval_transform = transforms.Compose([
  transforms.Resize((IMG_SIZE, IMG_SIZE)),
  transforms.ToTensor(),
  transforms.Normalize(mean=MEAN, std=STD),
])

In [ ]:
# Read dataset, convert image to RGB, applies the chosen transform and returns the image and its label
class RivalImageDataset(Dataset):
  def __init__(self, df, transform = None, is_test = False, return_mask = False):
    self.df = df.reset_index(drop=True)
    self.rival10_root = "/root/.cache/kagglehub/competitions/fall-2026-ecse-551-task-1/archive/RIVAL10_Kaggle"
    self.transform = transform
    self.is_test = is_test
    self.return_mask = False

  def __len__(self):
    return len(self.df)

  def resolve_image_path(self, raw_path):
    # Remove any leading slash so os.path.join joins cleanly
    rel_path = str(raw_path).removeprefix("RIVAL10").lstrip("/")
    return os.path.join(self.rival10_root, rel_path)

  def resolve_mask_path(self, raw_path):
    filename = os.path.basename(str(raw_path))
    return os.path.join(self.rival10_root, "ordinary", "entire_object_masks", filename)

  def __getitem__(self, index):
    row = self.df.iloc[index]
    img_path = self.resolve_image_path(row["image_path"])

    # force 3-channel RGB so grayscale/RGBA files match expected shape
    with Image.open(img_path) as img:
        image = img.convert("RGB")

    if self.transform is not None:
        image = self.transform(image)
    mask_tensor = None
    if self.return_mask:
      mask_path = self.resolve_image_path(row["image_path"])
      with Image.open(mask_path) as mask:
        mask = mask.convert("L") # convert to grayscale
        if isinstance(image.torch.Tensor):
          _,h,w = image.shape
          mask = mask.Resize((w,h), resample=Image.NEAREST)
          mask_tensor = torch.from_numpy((np.array(mask) > 127).astype(np.float32)) # cast bright pixels to 1.0 and darker pixels to 0.0

    if self.is_test:
      if self.return_mask:
        return image, mask_tensor, row["image_path"]
      return image, row["image_path"]

    label = torch.tensor(int(row["label"]), dtype=torch.long)
    return image, label


# Build datasets for each split
train_dataset = RivalImageDataset(training_df, transform=train_transform_light, is_test = False, return_mask = False)
val_dataset = RivalImageDataset(val_df, transform=eval_transform, is_test = False, return_mask = False)
local_test_dataset = RivalImageDataset(local_test_df, transform=eval_transform, is_test = False, return_mask = False)
kaggle_test_dataset = RivalImageDataset(test_df, transform=eval_transform, is_test = True, return_mask = False)
mask_eval_dataset = RivalImageDataset(local_test_df, transform = eval_transform, is_test = False, return_mask = True) # to experimment with object vs background

# Wrap in DataLoaders (shuffle=True only for training)
BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
local_test_loader = DataLoader(local_test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
kaggle_test_loader = DataLoader(kaggle_test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
mask_eval_loader = DataLoader(mask_eval_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

CNN

In [ ]:
class CNN(nn.Module):

  def __init__(self, num_classes = 10, negative_slope = 0.01, dropout_p = 0.3):
    super().__init__()
    self.features = nn.Sequential(
        # layer 1 from (3,224,224) to (64,112,112) then pool (64,56,56)
        nn.Conv2d(3, 64, kernel_size = 7, stride = 2, padding = 3, bias = False),
        nn.BatchNorm2d(64),
        nn.LeakyReLU(negative_slope = negative_slope, inplace = True),
        nn.MaxPool2d(kernel_size = 2, stride = 2),
        # layer 2 from (64, 56, 56) to (128, 28, 28)
        nn.Conv2d(64, 128, kernel_size = 5, padding = 2, bias = False),
        nn.BatchNorm2d(128),
        nn.LeakyReLU(negative_slope = negative_slope, inplace = True),
        nn.MaxPool2d(kernel_size = 2, stride = 2),
        # layer 3 from (128, 28, 28) to (256, 14, 14)
        nn.Conv2d(128, 256, kernel_size = 3, padding = 1, bias = False),
        nn.BatchNorm2d(256),
        nn.LeakyReLU(negative_slope = negative_slope, inplace = True),
        nn.MaxPool2d(kernel_size = 2, stride = 2)
    )

    # Condense from (256, 14, 14) to (256, 4, 4)
    self.global_pool = nn.AdaptiveAvgPool2d(4)

    # Classification Step
    self.classifier = nn.Sequential(
        nn.Flatten(),
        nn.Linear(256 * 4 * 4,256),
        nn.LeakyReLU(negative_slope=negative_slope, inplace=True),
        nn.Dropout(p = dropout_p),
        nn.Linear(256, num_classes)
    )

  def forward(self,x):
    x = self.features(x)
    x = self.global_pool(x)
    x = self.classifier(x)
    return x

In [ ]:
if torch.cuda.is_available():
  device = torch.device("cuda")
else:
  device = torch.device("cpu")

# Hyperparams

epochs = 1
lr = 1e-3
weight_decay = 1e-4
patience = epochs # TO CHANGE LATER  # for early stopping we stop after 5 epochs if no meaningful validation_loss improvement
min_delta = 1e-4 # minimum decrease in validation_loss
best_weights_path = "cnn_best_weights.pth"

model = CNN(num_classes= 10, negative_slope= 0.01, dropout_p= 0.3).to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr = lr, weight_decay = weight_decay)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

torch.backends.cudnn.benchmark = True
model = model.to(device).to(memory_format=torch.channels_last)
scaler = torch.amp.GradScaler("cuda")



def evaluate(model, loader, loss_fn, device):
  model.eval()
  total_loss = 0.0
  correct = 0
  total = 0

  with torch.no_grad():
    for images, labels in loader:
      images = images.to(device, non_blocking = True)
      labels = labels.to(device, non_blocking = True)
      with torch.autocast(device_type="cuda", dtype=torch.float16):
        logits = model(images)
        loss = loss_fn(logits, labels) # predicted vs true

      total_loss += loss.item() * images.size(0)
      predictions = logits.argmax(dim=1)
      correct += (predictions == labels).sum().item() # how many predicted correctly
      total += labels.size(0)

  average_loss = total_loss / total
  accuracy = correct / total
  return average_loss, accuracy

# Training 

best_validation_loss = float("inf")
best_weights = None
epochs_without_improvement = 0
history = {"training_loss": [], "training_accuracy": [], "validation_loss":[], "validation_accuracy": [], "learning_rate": []}

for epoch in range(1, epochs + 1):
  model.train()
  current_loss = 0.0
  current_correct = 0.0
  current_total = 0.0

  for images, labels in train_loader:
    images = images.to(device, non_blocking = True).to(memory_format=torch.channels_last)
    labels = labels.to(device, non_blocking = True)

    optimizer.zero_grad()
    with torch.autocast(device_type="cuda", dtype=torch.float16):
      logits = model(images)
      loss = loss_fn(logits, labels)

    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()

    current_loss += loss.item() * images.size(0)
    predictions = logits.argmax(dim=1)
    current_correct += (predictions == labels).sum().item()
    current_total += labels.size(0)
  
  training_loss = current_loss / current_total
  training_accuracy = current_correct / current_total
  validation_loss, validation_accuracy = evaluate(model, val_loader, loss_fn, device)

  history["training_loss"].append(training_loss)
  history["training_accuracy"].append(training_accuracy)
  history["validation_loss"].append(validation_loss)
  history["validation_accuracy"].append(validation_accuracy)
  history["learning_rate"].append(optimizer.param_groups[0]["lr"])

  scheduler.step()

  print(
      f"Epoch {epoch:02d}/{epochs:02d} | "
      f"Train Loss: {training_loss:.4f}, Acc: {training_accuracy:.4f} | "
      f"Val Loss: {validation_loss:.4f}, Acc: {validation_accuracy:.4f}"
  )

  if validation_loss < (best_validation_loss - min_delta):
    best_validation_loss = validation_loss
    best_weights = copy.deepcopy(model.state_dict())
    torch.save(best_weights, best_weights_path)
    epochs_without_improvement = 0 # reset to 0
  else:
    epochs_without_improvement += 1
    if epochs_without_improvement >= patience:
      print("Early stopping triggered")
      break

# Restore the best weights!!!

if best_weights is not None:
  model.load_state_dict(best_weights)

